# Importación de Librerías

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')

# Carga de datos y del modelo final

In [ ]:
X_test = pd.read_parquet('../data/processed/X_test.parquet')
y_test = pd.read_parquet('../data/processed/y_test.parquet')

model = joblib.load('../outputs/modelo_final.pkl')
y_pred = model.predict(X_test)


[0. 0. 0. 0.]


# Cálculo de valores SHAP  e interpretación

Ahora realizaremos la importación de los datos de prueba (X_test e y_test), cargaremos el modelo entrenado previamente mediante joblib y generaremos las predicciones (y_pred) con el método predict. A continuación, inicializaremos el objeto TreeExplainer de SHAP optimizado para modelos de árboles, seleccionaremos una muestra aleatoria de hasta 3,000 filas de los datos de prueba (X_test) para optimizar el rendimiento y los tiempos de cálculo, y finalmente calcularemos los valores SHAP (shap_values) para interpretar y explicar las decisiones del modelo. 
SHAP se utiliza para calcular de manera matemática y justa cuánto contribuyó (positiva o negativamente) cada variable a las predicciones del modelo, tanto a nivel general (global) como para casos específicos (local).

In [ ]:
explainer = shap.TreeExplainer(model)
X_shap_sample = X_test.sample(n=min(3000, len(X_test)), random_state=42)
shap_values = explainer(X_shap_sample)

A continuación se hace un bosquejo visual de los resutlados obtenidos:

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap_sample, show=False)
plt.title("Impacto Global de Variables (SHAP Summary Plot)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png', bbox_inches='tight', dpi=300)
plt.show()

Dependencia SHAP

In [ ]:
plt.figure(figsize=(8, 5))
shap.dependence_plot("trip_distance", shap_values.values, X_shap_sample, interaction_index="hour", show=False)
plt.title("Dependencia SHAP: Distancia del Viaje", fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/shap_dependence_distancia.png', bbox_inches='tight', dpi=300)
plt.show()


Análisis de Errores por Subgrupos

In [ ]:
df_eval = X_test.copy()
df_eval['y_true'] = y_test.values
df_eval['y_pred'] = y_pred
df_eval['error_absoluto'] = np.abs(df_eval['y_true'] - df_eval['y_pred'])

# Segmentación por subgrupos críticos
is_airport = df_eval.get('DOLocationID', pd.Series([False]*len(df_eval))).isin([132, 138, 1]) | \
             df_eval.get('PULocationID', pd.Series([False]*len(df_eval))).isin([132, 138, 1])

is_madrugada = df_eval.get('hour', pd.Series([0]*len(df_eval))).between(0, 5)

umbral_largo = df_eval['trip_distance'].quantile(0.90)
is_largo = df_eval['trip_distance'] > umbral_largo

print(f"MAE General: {df_eval['error_absoluto'].mean():.2f} min")
print(f"MAE Aeropuertos: {df_eval[is_airport]['error_absoluto'].mean():.2f} min")
print(f"MAE Madrugada: {df_eval[is_madrugada]['error_absoluto'].mean():.2f} min")
print(f"MAE Viajes Largos: {df_eval[is_largo]['error_absoluto'].mean():.2f} min")


# Gráfico de Errores

In [ ]:
subgrupos = ['General', 'Aeropuertos', 'Madrugada', 'Viajes Largos']
maes = [
    df_eval['error_absoluto'].mean(),
    df_eval[is_airport]['error_absoluto'].mean(),
    df_eval[is_madrugada]['error_absoluto'].mean(),
    df_eval[is_largo]['error_absoluto'].mean()
]

plt.figure(figsize=(8, 5))
plt.bar(subgrupos, maes, color=['skyblue', 'salmon', 'orange', 'purple'])
plt.ylabel('Error Absoluto Medio (MAE en min)')
plt.title('Comparación de Error del Modelo por Subgrupos Críticos')
plt.ylim(0, max(maes) * 1.2)

for i, v in enumerate(maes):
    plt.text(i, v + 0.5, f"{v:.2f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/error_por_subgrupos.png', bbox_inches='tight', dpi=300)
plt.show()
